In [3]:
import os 
from dotenv import load_dotenv

load_dotenv()


True

## PART 1 — Groq Setup & Basic Chat

**Task 1: ChatGroq Setup**
1. Create a Groq account and generate an API key.
2. Configure the API key using environment variables.
3. Initialize ChatGroq using LangChain.

In [4]:
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")
else:
    print("GROQ_API_KEY is set")

GROQ_API_KEY is set


In [5]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
)


**Task 2 — Basic Chat with ChatGroq**

1. Send a simple prompt to ChatGroq.
2. Print the response.
3. Verify low-latency behavior.

In [6]:
res = llm.invoke("What is the capital of France?")
print(res.content)


The capital of France is **Paris**.


## PART 2 — RAG Backend Pipeline

**Task 3: Document Loading & Text Splitting**

1. Load documents using LangChain loaders (PDF/Text).
2. Split documents using RecursiveCharacterTextSplitter.
3. Configure:
   - chunk_size
   - chunk_overlap


In [11]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
loader = TextLoader("Book1.txt")
data = loader.load()
print(len(data))
print(data[0].page_content[:100])

1
/ 




THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say th


In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = splitter.split_documents(data)
print(len(all_splits))
print(all_splits[0].page_content[:100])

608
/ 




THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say th


**Task 4: Embeddings & Vector Store**

1. Generate embeddings (OpenAI / HuggingFace / Ollama embeddings).
2. Store embeddings in:
   - FAISS or ChromaDB
3. Create a retriever from the vector store.


In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)

In [15]:
retriever = vectorstore.as_retriever()

**Task 5: RAG Prompt Template**
Create a ChatPromptTemplate that:

- Injects retrieved context
- Accepts user question
- Instructs the model to answer only from context

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
system_template = ChatPromptTemplate.from_template(
     """
        You are a helpful assistant that can answer questions about the web page.
        You are given a question and a context.
        You need to answer the question based on the context.
        Answer ONLY from the context, else say you dont know.
        ---
        Context:
        {context}
        ---
        Question:
        {question}
        ---
    """
)




## PART 3 — ChatGroq RAG Chain

**Task 6: Build RAG Chain**
Create a pipeline: User Question → Retriever → Context → Prompt → ChatGroq → Answer
Test the chain with multiple questions.

In [19]:
def format_docs(docs):
    return "\n\n".join([
        f"Document {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])


In [20]:
chain = {
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
} | system_template | llm | StrOutputParser()

In [21]:

chain.invoke("What is Hogwarts?")

'Hogwarts is a school of witchcraft and wizardry.'

## PART 4 — Streamlit UI

**Task 7: Build Streamlit Chat Interface**
Create a Streamlit app with:

1. File uploader (PDF/Text)
2. Chat input box
3. Chat message display (user & AI)
4. Session state to store chat history

**Done in `app.py`**
- `st.file_uploader` for pdf/txt → save under `uploads/`
- `st.chat_input` + `st.chat_message` for the chat UI
- `st.session_state.messages` + `st.session_state.chat_store` for history across reruns

```bash
cd genai/langchain/assignment-30
streamlit run app.py
```


**Task 8: Integrate RAG Backend with UI**

1. Connect Streamlit UI with RAG chain.
2. On user query:
   - Retrieve relevant documents
   - Generate answer using ChatGroq
3. Display responses in chat format.

**How it works in `app.py`**
- upload → split → OpenAI embeddings → Chroma retriever
- history-aware standalone question (if chat_history exists) → retrieve → grounded Groq answer
- wrap with `RunnableWithMessageHistory` + `session_id` in config


## PART 5 — Testing & Validation

**Task 9: Multi-Turn Chat Testing**
Test the chatbot with:

1. Initial factual question — e.g. "What is Hogwarts?" / "Where does Harry live?"
2. Follow-up — e.g. "Who tells him he is a wizard?"
3. Out-of-context — e.g. "What is the capital of France?" → should say it doesnt know from context

Verify:
- answers stay grounded in `Book1.txt` / uploaded doc
- chat history still works for follow-ups in the Streamlit session


**Task 10 (UI checklist)**
- upload file → chunks + vectorstore created
- ask a doc question → grounded answer
- ask follow-up → still coherent
- ask unrelated question → "I don't know based on the provided context."


**Task 11: Observations & Insights**
Write short answers:

1. Why Groq is suitable for RAG chatbots → RAG already adds retrieve latency; Groq’s fast inference keeps the chat feeling snappy in a Streamlit UI. good fit when you want grounded answers without waiting forever on the LLM step.
2. Difference between Groq RAG and OpenAI RAG → same RAG pattern (embed → retrieve → prompt). difference is mostly the *generator*: Groq hosts fast open models (here `openai/gpt-oss-20b`); OpenAI uses their chat models. embeddings here still used OpenAI (`text-embedding-3-small`) either way.
3. Role of Streamlit in rapid GenAI prototyping → tiny amount of UI code: upload + chat widgets + session state. enough to demo the full RAG loop without building a real frontend/backend first.


In [ ]:
# Task 11 answers
print("1. Groq → fast LLM step, so RAG chat UI feels realtime.")
print("2. Groq vs OpenAI RAG: same retrieve+ground pattern; swap the chat model (embeddings can stay OpenAI).")
print("3. Streamlit → quick upload/chat prototype without a full web stack.")


In [ ]:
# (end)
